# VisionBench: Video Benchmarking Guide

A comprehensive guide for running video benchmarking on custom datasets.

## Why VisionBench?

**VisionBench** is a data-to-publication platform that transforms video foundation model benchmarking from months of work into days.

### Key Benefits

- ⚡ **Save months of debugging time** - Robust, tested infrastructure handles the complexity
- 🚀 **Rapidly produce publication-ready results** - From data to paper in a streamlined workflow
- 🔄 **Reproducible and scalable experiments** - Parallel processing on large datasets
- 📊 **Comprehensive evaluation suite** - Built-in metrics and custom metric support
- 🔌 **Modular architecture** - Easy setup of data pipelines, backbone models, classification heads, and evaluation metrics

### Architecture Overview

The system is organized with experimental configurations centralized in `./config`, while core module logic resides in `vision_bench`.

---

## Step 1: Dataset Organization

### File Structure

Organize your video files and annotations in the following format. Each `.mp4` file should be paired with a `.tsv` file containing frame-level labels:

```
base_dir/
├── video_001.mp4
├── video_001.tsv
└── ...
```

### Configuration

Configure your datasets in `config/data/datasets.py` with the following parameters:

1. **Base directory**: Source directory for your data (`base_dir`)
2. **Precomputed directory**: Destination directory for precomputed features

**Note**: Partition each video and label into reasonable sizes to leverage parallel processing for feature extraction and storage. See class comments for additional parameter details.

In [1]:
from config.data.datasets import CORALCAM
CORALCAM.path

'/share/j_sun/jth264/refactor/coralcam'

---

## Step 2: Video Preprocessing

Preprocess videos using your selected sliding window styles. This creates (frame, label) pairs stored in the precomputed directory.

In [2]:
from vision_bench.execution.preprocessor import Preprocessor

SLIDING_STYLES = [
    "frames_w_temp", 
    "sliding_window_w_temp", 
    "test_frames", 
    "test_sliding_window", 
]

Preprocessor(
    datasets=[CORALCAM],
    sliding_styles=SLIDING_STYLES,
    parallel=False,
).run(save_input=True)

FileNotFoundError: Precomputed path does not exist: /share/j_sun/jth264/refactor/coralcam/precomputed/train/GX017050_115/sliding_window_w_temp/inputs

---

## Step 3: Feature Extraction

Extract model features using your chosen backbone models. Features are stored alongside labels and inputs.

### Resumable Extraction

For large datasets where extraction may be interrupted (e.g., preempted jobs), the validator class checks each data shard for the expected number of frames. The feature extractor can then skip completed subsets automatically.

In [ ]:
from vision_bench.execution.feature_extractor import FeatureExtractor

TARGET_MODELS = [
    "videomae",
    "resnet50", 
    "dinov3_large",
]

REPORT_ROOT = "data/validation/reports"

FeatureExtractor(
    datasets=[CORALCAM],
    sliding_styles=SLIDING_STYLES,
    models=TARGET_MODELS,
).set_default_validator(validator_root=REPORT_ROOT).run()

---

## Step 4: Data Validation *(Optional)*

If feature extraction ran on a subset of your corpus, use the data validator to identify completed subsets and rerun extraction only where needed.

In [ ]:
from vision_bench.execution.validator import Validator
Validator(
    datasets=[CORALCAM],
    sliding_styles=SLIDING_STYLES,
    root_path=REPORT_ROOT,
).run(models=TARGET_MODELS)

---

## Step 5: Model Training

### Configuration

Configure training settings in `config/experiments`. The Trainer passes these configurations to `vision_bench/scripts/train.py`.

**Customization**: To create custom experiment configurations:
1. Modify experiment type definitions in `vision_bench/typing/experiment.py`
2. Update training scripts to interpret new elements

VisionBench provides comprehensive default training configurations including optimizer, learning rate, epochs, and more.

### Run Management

The `WandbRunMatcher` manages experiment-to-run correspondence and identifies which experiments haven't been executed yet.

In [ ]:
from config.experiments.cvpr import CVPR_EXPS
from vision_bench.execution.trainer import Trainer
from vision_bench.management.wandb_matcher import WandbRunMatcher

ENTITY = "fish_benchmark"
TRAINING_PROJECT = CORALCAM.name
ARTIFACT_DIR = "./checkpoints"

Trainer(
    CVPR_EXPS, 
    WandbRunMatcher(ENTITY, TRAINING_PROJECT), 
    local_artifact_dir=ARTIFACT_DIR
).run()

---

## Step 6: Model Evaluation

After training completes, evaluate model performance on the test set. The `WandbMatcher` automatically:
- Identifies completed training runs
- Retrieves the best model checkpoint

The evaluator produces full model output logits with corresponding labels, facilitating custom metric calculations in the next step.

In [ ]:
from vision_bench.execution.evaluator import Evaluator

EVALUATION_PROJECT = f"{TRAINING_PROJECT}_eval"
PREDICTIONS_DIR = "./predictions"
Evaluator(
    CVPR_EXPS, 
    WandbRunMatcher(ENTITY, TRAINING_PROJECT), 
    WandbRunMatcher(ENTITY, EVALUATION_PROJECT),
    model_ckpt_dir=ARTIFACT_DIR, 
    local_artifact_dir=PREDICTIONS_DIR
).run()

---

## Step 7: Results Export

Export results using custom metrics based on predictions from the evaluation step.

### Metrics

Any callable function on predictions and targets can serve as a metric. VisionBench provides a comprehensive default metric suite in `config.metrics.metrics`.

In [ ]:
from config.metrics.metrics import * 
from vision_bench.execution.exportor import Exportor

OUTPUT_PATH = "./results"

# Aggregate metrics
aggregate_metrics = {
    "f1_micro": F1Micro(),
    "f1_macro": F1Macro(),
    "precision_micro": PrecisionMicro(),
    "precision_macro": PrecisionMacro(),
    "recall_micro": RecallMicro(),
    "recall_macro": RecallMacro(),
    "acc": Accuracy(),
    "mAP": mAP(),
}

# Per-class metrics
per_class_metrics = {
    "f1_per_class": F1PerClass(),
    "precision_per_class": PrecisionPerClass(),
    "recall_per_class": RecallPerClass(),
    "positive_per_class": PositivePerClass(),
    "ap_per_class": APPerClass(),
    "confusion_matrix": binary_confusion_matrix,
}

# Subgroup definitions
subgroup_mapping = {
    "biting": [0, 1], 
    "aggression": [3]
}

Exportor(
    experiments=CVPR_EXPS,
    train_matcher=WandbRunMatcher(ENTITY, TRAINING_PROJECT),
    eval_matcher=WandbRunMatcher(ENTITY, EVALUATION_PROJECT), 
    aggregate_metrics=aggregate_metrics,
    per_class_metrics=per_class_metrics,
    subgroup_mappings=subgroup_mapping, 
    output_base=OUTPUT_PATH,
    output_name=f"{CORALCAM.name}_results.csv"
).run()

---

## Conclusion

You now have a complete end-to-end research pipeline for benchmarking vision foundation models on large-scale parallel datasets. Start benchmarking today!